# Multimodal AI Model Exploration
## Assignment 3 — OpenAI API

This notebook demonstrates **7 input-output modalities** using OpenAI APIs.

| # | Modality | Model |
|---|----------|-------|
| 1 | Text → Text | `gpt-4o-mini` |
| 2 | Text → Image | `dall-e-3` |
| 3 | Image → Text | `gpt-4o` (vision) |
| 4 | Text → Audio | `tts-1-hd` |
| 5 | Audio → Text | `whisper-1` |
| 6 | Text → Video | `dall-e-3` frames → animated GIF |
| 7 | Video → Text | `gpt-4o` (vision + frame extraction) |

In [ ]:
!pip install openai pillow requests opencv-python-headless numpy -q

In [ ]:
import base64
import io
import os
import requests
import numpy as np
from PIL import Image as PILImage
from IPython.display import display, Image, Audio, HTML
from openai import OpenAI

OPENAI_API_KEY = "your-openai-api-key-here"

client = OpenAI(api_key=OPENAI_API_KEY)
print("Client initialized successfully!")

---
## 1. Text → Text
**Model:** `gpt-4o-mini`  
Send a text prompt and receive a text response.

In [ ]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a concise and helpful AI assistant."},
        {"role": "user", "content": "Explain the concept of neural networks in exactly 3 sentences."},
    ],
)

print("=== TEXT → TEXT (gpt-4o-mini) ===")
print(response.choices[0].message.content)

---
## 2. Text → Image
**Model:** `dall-e-3`  
Generate an image from a text description.

In [ ]:
response = client.images.generate(
    model="gpt-image-1",
    prompt="A futuristic underwater city with bioluminescent buildings and schools of colorful fish swimming between skyscrapers, cinematic lighting, 4K",
    size="1024x1024",
    quality="medium",
    n=1,
)

# gpt-image-1 returns base64, not a URL
img_bytes = base64.b64decode(response.data[0].b64_json)

with open("generated_image.png", "wb") as f:
    f.write(img_bytes)

print("=== TEXT → IMAGE (gpt-image-1) ===")
print("Saved as: generated_image.png")
display(Image("generated_image.png"))

---
## 3. Image → Text
**Model:** `gpt-4o` (vision)  
Feed an image to the model and get a text description.

> Uses the image generated in the previous cell.

In [ ]:
with open("generated_image.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe this image in detail. What objects, colors, and mood do you notice?"},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        ],
    }],
)

print("=== IMAGE → TEXT (gpt-4o vision) ===")
print(response.choices[0].message.content)

---
## 4. Text → Audio
**Model:** `tts-1-hd`  
Convert text into spoken audio using OpenAI's high-definition TTS.

In [ ]:
text_to_speak = (
    "Welcome to the world of multimodal artificial intelligence! "
    "Today we're exploring how AI can understand and generate text, images, audio, and video. "
    "This audio was created using OpenAI's text-to-speech model, tts-1-hd."
)

response = client.audio.speech.create(
    model="tts-1-hd",
    voice="nova",
    input=text_to_speak,
)

response.stream_to_file("generated_audio.mp3")

print("=== TEXT → AUDIO (tts-1-hd, voice: nova) ===")
print(f"Input text: {text_to_speak[:80]}...")
print("Saved as: generated_audio.mp3")
display(Audio("generated_audio.mp3"))

---
## 5. Audio → Text
**Model:** `whisper-1`  
Transcribe the audio generated in the previous cell back to text.

In [ ]:
with open("generated_audio.mp3", "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        response_format="text",
    )

print("=== AUDIO → TEXT (whisper-1) ===")
print(transcription)

---
## 6. Text → Video
**Model:** `dall-e-3` (sequential frames → animated GIF)  

OpenAI does not yet offer native video generation, so we generate multiple DALL-E 3 frames showing sequential stages of a scene and stitch them into an animated GIF.

In [ ]:
def text_to_video(prompt, n_frames=4, output_path="generated_video.gif"):
    """Generate an animated GIF from a text prompt using gpt-image-1 frames."""
    frames = []
    for i in range(n_frames):
        frame_prompt = f"{prompt}, moment {i+1} of {n_frames}, sequential progression"
        print(f"  Generating frame {i+1}/{n_frames}...")
        resp = client.images.generate(
            model="gpt-image-1",
            prompt=frame_prompt,
            size="1024x1024",
            quality="low",
            n=1,
        )
        img_bytes = base64.b64decode(resp.data[0].b64_json)
        img = PILImage.open(io.BytesIO(img_bytes)).resize((512, 512)).convert("RGB")
        frames.append(img)

    frames[0].save(
        output_path,
        save_all=True,
        append_images=frames[1:],
        optimize=False,
        duration=1500,
        loop=0,
    )
    return output_path


print("=== TEXT → VIDEO (gpt-image-1 frames → GIF) ===")
print("Generating 4 frames (this takes ~80s)...")
video_path = text_to_video("A rocket launching from Earth into outer space, reaching the Moon")
print(f"Saved as: {video_path}")

display(HTML(f'<img src="{video_path}" style="width:480px; border-radius:8px;" />'))

---
## 7. Video → Text
**Model:** `gpt-4o` (vision)  

Extract frames from the GIF generated above and send them to GPT-4o vision to produce a summary and description.

In [ ]:
def gif_frames_to_b64(gif_path, max_frames=5):
    """Extract evenly-spaced frames from a GIF and return them as base64 JPEG strings."""
    gif = PILImage.open(gif_path)
    all_frames = []
    try:
        while True:
            all_frames.append(gif.copy().convert("RGB"))
            gif.seek(gif.tell() + 1)
    except EOFError:
        pass
    indices = [int(i * len(all_frames) / max_frames) for i in range(min(max_frames, len(all_frames)))]
    frames_b64 = []
    for idx in indices:
        buf = io.BytesIO()
        all_frames[idx].save(buf, format="JPEG")
        frames_b64.append(base64.b64encode(buf.getvalue()).decode())
    return frames_b64


frames = gif_frames_to_b64("generated_video.gif")
print(f"Extracted {len(frames)} frames from the GIF.")

content = [{
    "type": "text",
    "text": f"These are {len(frames)} frames extracted from a video/animation. Describe what is happening step by step and provide an overall summary.",
}]
for b64 in frames:
    content.append({
        "type": "image_url",
        "image_url": {"url": f"data:image/jpeg;base64,{b64}"},
    })

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": content}],
)

print("=== VIDEO → TEXT (gpt-4o vision) ===")
print(response.choices[0].message.content)

---
## Summary

| Modality | Model | Status |
|----------|-------|--------|
| Text → Text | `gpt-4o-mini` | ✅ |
| Text → Image | `dall-e-3` | ✅ |
| Image → Text | `gpt-4o` (vision) | ✅ |
| Text → Audio | `tts-1-hd` | ✅ |
| Audio → Text | `whisper-1` | ✅ |
| Text → Video | `dall-e-3` frames → GIF | ✅ |
| Video → Text | `gpt-4o` (vision) | ✅ |

All modalities are demonstrated using **OpenAI** as the sole AI platform.